In [ ]:
# Import drive for files
from google.colab import drive
drive.mount('/content/drive')

# Installation
!pip -q install pymupdf pdfplumber pandas pylatexenc

# Packages
import re, json
from pathlib import Path
import fitz
import pdfplumber
import pandas as pd
from pylatexenc.latexencode import unicode_to_latex

# Directory for files and files saved
pdf_directory = Path("/content/drive/MyDrive/Math Olympiad Competition/svsu_pdfs")
pdf_files = sorted([p for p in pdf_directory.iterdir() if p.suffix.lower() == ".pdf"])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def id_function(i: int) -> str:
    # there are 26^3*100 = possible matches
    numbers = f"{i % 1000:03d}"
    x = i // 1000
    letters = "".join(chr(ord("a") + ((x // (26**j)) % 26)) for j in reversed(range(3)))
    return numbers + letters

# Latex conversion rules
latex_conversion = {
    "≤": r"\le", "≥": r"\ge", "≠": r"\ne", "≈": r"\approx",
    "×": r"\times", "·": r"\cdot", "÷": r"\div",
    "π": r"\pi", "θ": r"\theta", "∞": r"\infty",
    "ℝ": r"\mathbb{R}", "ℤ": r"\mathbb{Z}", "ℚ": r"\mathbb{Q}", "ℕ": r"\mathbb{N}",
    "∠": r"\angle", "…": r"\ldots",
}
# Suggested cleanups for latex
def cleanup_latex(s: str) -> str:
    s = re.sub(r"\\ensuremath\{([^{}]+)\}", r"\1", s)
    s = s.replace(r"{\textrightarrow}", r"\to").replace(r"{\textdegree}", r"^\circ")
    s = s.replace(r"{\textemdash}", "---").replace(r"{\textendash}", "--")
    s = s.replace(r"{\textquotedblleft}", '"').replace(r"{\textquotedblright}", '"')
    s = s.replace(r"{\textquoteleft}", "'").replace(r"{\textquoteright}", "'")
    s = s.replace(r"\ensuremath{\sqrt{}}", r"\sqrt")
    s = re.sub(r"(\\angle)([A-Z])", r"\1 \2", s)
    s = re.sub(r"(\\sqrt)\{\}\s*(\d+)", r"\1{\2}", s)
    s = re.sub(r"(\\sqrt)\{\}\s*([a-zA-Z])", r"\1{\2}", s)
    s = re.sub(r"\\sqrt(\d+)", r"\\sqrt{\1}", s)
    return s

# Clean text by applying two previous functions
def convert_text(t: str) -> str:
    for x, v in latex_conversion.items():
        t = t.replace(x, v)
    t = unicode_to_latex(t, non_ascii_only=True)
    t = cleanup_latex(t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

In [ ]:
# In here extract text with pymudpdf
def extract_with_pymupdf(pdf_path: Path) -> str:
    pages_text = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            blocks = page.get_text("blocks")
            blocks = sorted(blocks, key=lambda b: (b[1], b[0]))
            pages_text.append("\n".join(b[4].strip() for b in blocks if b[4].strip()))
    return "\n\n".join(pages_text)

# In here extract text with pdfplumber
def extract_with_pdfplumber(pdf_path: Path) -> str:
    pages = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            pages.append(page.extract_text(layout=True) or "")
    return "\n\n".join(pages)
# Try PyMuPDF first if extraction is very short or fails fall back to pdfplumber
def extract_text_from_pdf(pdf_path: Path) -> str:
    try:
        txt = extract_with_pymupdf(pdf_path)
        if len(txt.strip()) > 200:
            return txt
    except Exception as e:
        print(f"[warn] PyMuPDF failed on {pdf_path.name}: {e}")
    try:
        return extract_with_pdfplumber(pdf_path)
    except Exception as e:
        print(f"[error] pdfplumber failed on {pdf_path.name}: {e}")
        return ""

In [ ]:
# Find problem number sign, and solution indicator
PROB_RE = re.compile(r"(?m)^\s*(\d{1,2})[\)\.]\s+")
SOL_RE  = re.compile(r"(?mi)^\s*Solution\s*:?\s*\(?\s*([A-E])?\s*\)?\s*")

# Split the full extracted text into per-problem chunks using PROB_RE marks
def split_problem_chunks(txt: str):
    m = list(PROB_RE.finditer(txt))
    out = []
    for i, match in enumerate(m):
        qnum = int(match.group(1))
        start = match.end()
        end = m[i+1].start() if i+1 < len(m) else len(txt)
        out.append((qnum, txt[start:end].strip()))
    return out

# Within a single problem chunk, find the "Solution" marker
def split_problem_solution(chunk: str):
    sm = SOL_RE.search(chunk)
    if not sm:
        return None
    ans = sm.group(1)
    return chunk[:sm.start()].strip(), chunk[sm.end():].strip(), ans

#Output paths: one CSV for easy inspection and one JSONL for fine-tuning-style records
csv_out = Path("/content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/solved_only.csv")
jsonl_out = csv_out.with_suffix(".jsonl")
csv_out.parent.mkdir(parents=True, exist_ok=True)

# System prompt used for JSONL training
SYSTEM = "You are a math olympiad tutor. Solve carefully. Use clear reasoning and LaTeX when helpful."
# Minimum lengths to filter out tiny/garbled extracts (often headers/footers or incomplete solutions).
MIN_PROB_CHARS, MIN_SOL_CHARS = 20, 60

rows = []
for pdf_order, pdf_path in enumerate(pdf_files):
    # Extract raw text from PDF PyMuPDF first, fallback to pdfplumber if need be
    raw = extract_text_from_pdf(pdf_path)
    # If too little text remove
    if len(raw.strip()) < 200:
        continue
    # Split into numbered problem blocks within this PD
    for q_order, (qnum, chunk) in enumerate(split_problem_chunks(raw)):
        # Split each block into (problem, solution) by locating a "Solution" header
        ps = split_problem_solution(chunk)
        if not ps:
            continue
        prob_raw, sol_raw, ans = ps
        # Normalize text and clean up spacing.
        prob, sol = convert_text(prob_raw), convert_text(sol_raw)
        # Filter out too-short problems/solutions (often extraction noise) ?????? INVESTIGATE FURTHER IF THIS IS ACTUALLY NEEDED
        if len(prob) < MIN_PROB_CHARS or len(sol) < MIN_SOL_CHARS:
            continue
        # save rows
        rows.append({"pdf_order": pdf_order, "q_order": q_order, "pdf": pdf_path.name,
                     "qnum": qnum, "answer_choice": ans, "problem": prob, "solution": sol})

# Sort by file order then within-file order to keep stable IDs and consistent output
rows.sort(key=lambda r: (r["pdf_order"], r["q_order"]))
# Assign ID
for i, r in enumerate(rows):
    r["id"] = id_function(i)
# Build a DataFrame for CSV export and drop columns not needed
df = pd.DataFrame(rows).drop(columns=["pdf_order", "q_order"])
df.to_csv(csv_out, index=False)
# --------------------------------------------------------------------------------- (Examine further)
# Write JSONL where each line is one training-style conversation:
# system prompt + user problem (+ instruction if multiple choice) + assistant solution (prefixed with answer if present).
with open(jsonl_out, "w", encoding="utf-8") as f:
    for _, r in df.iterrows():
        user = r["problem"]
        assistant = r["solution"]
        # If we captured an answer letter, encourage the model to output the final answer as A–E.
        if pd.notna(r["answer_choice"]) and r["answer_choice"]:
            user += "\n\nReturn the final answer as a letter (A–E) and explain your reasoning."
            assistant = f"Answer: {r['answer_choice']}\n\n{assistant}"
        f.write(json.dumps({
            "id": r["id"],
            "messages": [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": user},
                {"role": "assistant", "content": assistant},
            ],
            "meta": {"pdf": r["pdf"], "qnum": int(r["qnum"])}
        }, ensure_ascii=False) + "\n")

print("CSV Done:", csv_out, "rows:", len(df))
print("JSON Done:", jsonl_out, "rows:", len(df))
display(df.head(5))

[error] pdfplumber failed on 2008report.pdf: No /Root object! - Is this really a PDF?


CSV Done: /content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/solved_only.csv rows: 681
JSON Done: /content/drive/MyDrive/Math Olympiad Competition/QuestionsCSV/solved_only.jsonl rows: 681


,pdf,qnum,answer_choice,problem,solution,id
0,2003Level1_Solutions.pdf,1,C,1\n1\n9\n16\n+\n=\na. 1 \n \nb. \n5\n7 \ne. No...,": 1\n16\n9\n144\n=\nand 1\n9\n16\n144\n=\n, so...",000aaa
1,2003Level1_Solutions.pdf,2,B,Which of these numbers is largest?\na. \n3 5 6...,: Raise each to the 6th power. Since \n if and...,001aaa
2,2003Level1_Solutions.pdf,3,A,Last year a bicycle cost $160 and a cycling he...,: The bicycle went up to 160\n0.05(160)\n168\n...,002aaa
3,2003Level1_Solutions.pdf,4,C,A vacuum pump removes {\textonehalf} of the ai...,: The percentage goes from 100% to 50% to 25% ...,003aaa
4,2003Level1_Solutions.pdf,5,B,"The ratio of \nto \nw\nx is 4:3, of \n to \nis...",: \n4 6 2\n16\n3 1 3\n3\nw\nw x\nz\ny\nx\nz\ny...,004aaa
